# Atom Pair fingerprints for Data 1.1

## 1. Data ingestion and cleaning

In [ ]:
import pandas as pd

df_cdd = pd.read_csv('data_1.csv')
df_cdd = df_cdd.rename(columns={
    "GU 10 µM L6: Average Glucose uptake / isoprenaline (%)": "gu",
    "B2 cAMP L6: Average cAMP / isoprenaline (AU)": "camp",
})

In [ ]:
df_cdd = df_cdd.drop(['Synonyms',
                      'GU 10 µM L6: Average Glucose uptake / isoprenaline (%) Standard Deviation (±)',
                      'GU 10 µM L6: Average Glucose uptake / isoprenaline (%) Count',
                      'B2 cAMP L6: Average cAMP / isoprenaline (AU) Standard Deviation (±)',
                      'B2 cAMP L6: Average cAMP / isoprenaline (AU) Count'
                    ], axis=1)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from copy import deepcopy

# RDKit
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.AtomPairs import Pairs

# Sklearn
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# XGBoost
import xgboost as xgb

# SHAP
import shap

# PyTorch & PyG
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print('All imports successful.')

In [ ]:
df = df_cdd
df = df.dropna(subset=['camp'])
df.columns = df.columns.str.strip()

# Basic sanity check
required = {'Molecule Name', 'SMILES', 'gu', 'camp'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'

print(f'Dataset shape: {df.shape}')
print(df[['gu','camp']].describe())
df.head()

## 2. Atom Pair Fingerprints

> Atom Pair fingerprints encode pairs of atoms and the shortest-path distance between them. Each bit represents a unique (atom-type, distance, atom-type) triplet. RDKit returns a hashed, count-based sparse vector that is binarised here for consistency with the rest of the pipeline.

In [ ]:
AP_NBITS = 2048

def smiles_to_atom_pair(smi, n_bits=AP_NBITS):
    mol = Chem.MolFromSmiles(smi)
    fp  = rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=n_bits)
    return np.array(fp)

X = np.vstack(df['SMILES'].apply(smiles_to_atom_pair).values)  # (N, 2048)
y = df[['gu', 'camp']].values   # (N, 2)

# Atom Pair bit names for later labelling
ap_names = [f'AP_{i}' for i in range(AP_NBITS)]

# Remove always-zero or always-one bits (no variance)
var_mask = (X.var(axis=0) > 0)
X_filtered = X[:, var_mask]
names_filtered = [ap_names[i] for i, v in enumerate(var_mask) if v]

# Store original bit indices so we can map back from filtered → full 2048-bit space
ap_original_indices = [i for i, v in enumerate(var_mask) if v]

print(f'Atom Pair shape (raw): {X.shape}')
print(f'Atom Pair shape (after removing zero-variance bits): {X_filtered.shape}')

# Visualise bit frequency
bit_freq = X_filtered.mean(axis=0)
plt.figure(figsize=(14,3))
plt.bar(range(len(bit_freq)), bit_freq, color='steelblue')
plt.xlabel('Atom Pair bit index (filtered)')
plt.ylabel('Frequency')
plt.title('Atom Pair Fingerprint Bit Frequencies across the Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Extract target variables
target_cols = ['gu', 'camp']
y = df[target_cols].values

print("Target variables shape:", y.shape)

# Visualize target distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes = axes.ravel()

for idx, col in enumerate(target_cols):
    axes[idx].hist(df[col], bins=20, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('target_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Cross validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def cv_metrics(y_true, y_pred, target_names=['gu','camp']):
    """Return per-target RMSE, MAE, R2."""
    results = {}
    for i, name in enumerate(target_names):
        results[name] = {
            'RMSE': np.sqrt(mean_squared_error(y_true[:,i], y_pred[:,i])),
            'MAE':  mean_absolute_error(y_true[:,i], y_pred[:,i]),
            'R2':   r2_score(y_true[:,i], y_pred[:,i])
        }
    return results

def print_cv_summary(all_fold_metrics, model_name):
    print(f'\n{'─'*55}')
    print(f'  {model_name} — 5-Fold CV Summary')
    print(f'{'─'*55}')
    for target in ['gu','camp']:
        rmse_vals = [m[target]['RMSE'] for m in all_fold_metrics]
        mae_vals  = [m[target]['MAE']  for m in all_fold_metrics]
        r2_vals   = [m[target]['R2']   for m in all_fold_metrics]
        print(f'  [{target.upper()}]  RMSE: {np.mean(rmse_vals):.4f}±{np.std(rmse_vals):.4f} | '
              f'MAE: {np.mean(mae_vals):.4f}±{np.std(mae_vals):.4f} | '
              f'R2: {np.mean(r2_vals):.4f}±{np.std(r2_vals):.4f}')
    print(f'{'─'*55}')

## 4. Machine Learning models 
### 4.1 Multi-Output SVR (with StandardScaler)

In [ ]:
svr_fold_metrics = []
svr_fold_models  = []   # store (scaler, model) per fold

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filtered)):
    X_tr, X_val = X_filtered[train_idx], X_filtered[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)

    # MultiOutputRegressor wraps SVR for 2-target regression
    base_svr = SVR(kernel='rbf', C=10, epsilon=0.1, gamma='scale')
    mo_svr   = MultiOutputRegressor(base_svr, n_jobs=-1)
    mo_svr.fit(X_tr_sc, y_tr)

    y_pred = mo_svr.predict(X_val_sc)
    metrics = cv_metrics(y_val, y_pred)
    svr_fold_metrics.append(metrics)
    svr_fold_models.append((scaler, mo_svr, train_idx, val_idx))
    print(f'Fold {fold+1}: GU_RMSE={metrics["gu"]["RMSE"]:.4f}  cAMP_RMSE={metrics["camp"]["RMSE"]:.4f}')

print_cv_summary(svr_fold_metrics, 'Multi-Output SVR')

### 4.2 Multi-output random forest

In [ ]:
rf_fold_metrics = []
rf_fold_models  = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filtered)):
    X_tr, X_val = X_filtered[train_idx], X_filtered[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    rf = RandomForestRegressor(
        n_estimators=300, max_features='sqrt',
        random_state=SEED, n_jobs=-1
    )
    # RandomForestRegressor natively handles multi-output
    rf.fit(X_tr, y_tr)

    y_pred = rf.predict(X_val)
    metrics = cv_metrics(y_val, y_pred)
    rf_fold_metrics.append(metrics)
    rf_fold_models.append((rf, train_idx, val_idx))
    print(f'Fold {fold+1}: GU_RMSE={metrics["gu"]["RMSE"]:.4f}  cAMP_RMSE={metrics["camp"]["RMSE"]:.4f}')

print_cv_summary(rf_fold_metrics, 'Multi-Output Random Forest')

### 4.3 Multi-output XGBoost

In [ ]:
xgb_fold_metrics = []
xgb_fold_models  = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filtered)):
    X_tr, X_val = X_filtered[train_idx], X_filtered[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    # XGBoost ≥ 2.0 supports multi-output natively with multi_strategy
    xgb_model = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        multi_strategy='multi_output_tree',   # native multi-output
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    y_pred = xgb_model.predict(X_val)
    metrics = cv_metrics(y_val, y_pred)
    xgb_fold_metrics.append(metrics)
    xgb_fold_models.append((xgb_model, train_idx, val_idx))
    print(f'Fold {fold+1}: GU_RMSE={metrics["gu"]["RMSE"]:.4f}  cAMP_RMSE={metrics["camp"]["RMSE"]:.4f}')

print_cv_summary(xgb_fold_metrics, 'Multi-Output XGBoost')

### 4.4 Model performance comparison

In [ ]:
def aggregate_cv(fold_metrics, model_name):
    rows = []
    for target in ['gu','camp']:
        for metric in ['RMSE','R2']:
            vals = [m[target][metric] for m in fold_metrics]
            rows.append({'Model': model_name, 'Target': target,
                         'Metric': metric, 'Mean': np.mean(vals), 'Std': np.std(vals)})
    return pd.DataFrame(rows)

results_df = pd.concat([
    aggregate_cv(svr_fold_metrics,  'SVR'),
    aggregate_cv(rf_fold_metrics,   'Random Forest'),
    aggregate_cv(xgb_fold_metrics,  'XGBoost'),
], ignore_index=True)

# Pivot for readability
pivot = results_df.pivot_table(index=['Model','Target'], columns='Metric', values=['Mean','Std'])
print(pivot.round(4))

# Bar chart with best model highlighted in green
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, target in zip(axes, ['gu', 'camp']):
    sub = results_df[(results_df['Target'] == target) & (results_df['Metric'] == 'RMSE')]

    # Identify best model (lowest RMSE)
    best_idx = sub['Mean'].idxmin()
    best_model = sub.loc[best_idx, 'Model']

    # Assign colors: green for best, blue for others
    colors = ['green' if m == best_model else 'steelblue' for m in sub['Model']]

    ax.bar(sub['Model'], sub['Mean'], yerr=sub['Std'], 
           capsize=5, color=colors, alpha=0.85)

    ax.set_title(f'{target.upper()} — Mean RMSE (5-Fold CV)')
    ax.set_ylabel('RMSE')
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


print('\nExpected output: bar chart + pivot table with RMSE/MAE/R2 for all 4 models on both targets.')

## 5. Helper — Bias Score for High GU + Low cAMP

The **bias score** for a feature $f$ across samples is:

$$\text{BiasScore}(f) = \phi_{gu}(f) - \phi_{camp}(f)$$

where $\phi$ are SHAP values. A positive bias score means the feature pushes toward **high gu AND low cAMP** — the desired combination.

In [ ]:
def bias_score_plot(shap_gu, shap_camp, feature_names, top_n=20, title='Bias Score'):
    """
    shap_gu, shap_camp: (N, F) SHAP value arrays
    bias = mean(shap_gu) - mean(shap_camp)
    Positive → feature promotes high GU and/or low cAMP
    """
    mean_gu   = shap_gu.mean(axis=0)
    mean_camp = shap_camp.mean(axis=0)
    bias = mean_gu - mean_camp

    idx_sorted = np.argsort(np.abs(bias))[::-1][:top_n]
    top_names  = [feature_names[i] for i in idx_sorted]
    top_bias   = bias[idx_sorted]

    colors = ['#2196F3' if b > 0 else '#F44336' for b in top_bias]
    plt.figure(figsize=(10, 6))
    plt.barh(top_names[::-1], top_bias[::-1], color=colors[::-1])
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Bias Score  (↑ favours High GU & Low cAMP)')
    plt.title(title)
    plt.tight_layout()
    plt.show()

    top_df = pd.DataFrame({'Feature': top_names, 'BiasScore': top_bias,
                           'MeanSHAP_gu': mean_gu[idx_sorted],
                           'MeanSHAP_camp': mean_camp[idx_sorted]})
    return top_df

print('Bias-score helper defined.')

## 6. Explainable AI methods
### 6.1 KernelSHAP — SVR

KernelSHAP is model-agnostic and the appropriate SHAP method for SVR.

In [ ]:
# Use best fold (lowest mean RMSE across targets)
best_fold_svr = np.argmin([
    (m['gu']['RMSE'] + m['camp']['RMSE']) / 2 for m in svr_fold_metrics
])
scaler_svr, mo_svr_best, tr_idx, val_idx = svr_fold_models[best_fold_svr]

X_tr_sc_svr  = scaler_svr.transform(X_filtered[tr_idx])
X_val_sc_svr = scaler_svr.transform(X_filtered[val_idx])

# KernelSHAP uses a background summary (k-means to keep it tractable)
background = shap.kmeans(X_tr_sc_svr, 30)

shap_svr_gu, shap_svr_camp = [], []

for i, estimator in enumerate(mo_svr_best.estimators_):
    print(f'  Computing KernelSHAP for SVR output {i+1}/2 ...')
    explainer = shap.KernelExplainer(estimator.predict, background)
    sv = explainer.shap_values(X_val_sc_svr, nsamples=200)
    if i == 0:
        shap_svr_gu = sv
    else:
        shap_svr_camp = sv

# ── Individual SHAP summary plots ─────────────────────────────────────────
print('\n[SVR] SHAP Summary — GU:')
shap.summary_plot(shap_svr_gu, X_val_sc_svr, feature_names=names_filtered,
                  max_display=20, plot_type='bar', show=True)

print('\n[SVR] SHAP Summary — cAMP:')
shap.summary_plot(shap_svr_camp, X_val_sc_svr, feature_names=names_filtered,
                  max_display=20, plot_type='bar', show=True)

# ── Bias score ─────────────────────────────────────────────────────────────
print('\n[SVR] Bias Score — High GU & Low cAMP:')
svr_bias_df = bias_score_plot(
    np.array(shap_svr_gu), np.array(shap_svr_camp),
    names_filtered, top_n=20, title='SVR — Bias Score (High GU & Low cAMP)'
)
display(svr_bias_df.head(10))

### 6.2 TreeSHAP — Random Forest

In [ ]:
best_fold_rf = np.argmin([
    (m['gu']['RMSE'] + m['camp']['RMSE']) / 2 for m in rf_fold_metrics
])
rf_best, tr_idx_rf, val_idx_rf = rf_fold_models[best_fold_rf]

X_tr_rf  = X_filtered[tr_idx_rf]
X_val_rf = X_filtered[val_idx_rf]

# ── (a) MDI Feature Importance ─────────────────────────────────────────────
# RF natively stores multi-output importance as the mean over estimators/outputs
fi = rf_best.feature_importances_   # shape (F,)

# Per-output importance by examining each tree's per-output contributions
# (sklearn averages them, so we replicate manually)
n_feats = X_tr_rf.shape[1]
fi_per_output = np.zeros((2, n_feats))
for tree in rf_best.estimators_:
    fi_per_output += tree.feature_importances_.reshape(1, n_feats)
fi_per_output /= len(rf_best.estimators_)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, target_i, target_name in zip(axes, [0,1], ['GU','cAMP']):
    top20_idx = np.argsort(fi_per_output[target_i])[::-1][:20]
    ax.barh([names_filtered[i] for i in top20_idx][::-1],
             fi_per_output[target_i][top20_idx][::-1], color='teal')
    ax.set_title(f'RF MDI Importance — {target_name}')
    ax.set_xlabel('Importance')
plt.suptitle('Random Forest Feature Importance (MDI)', fontweight='bold')
plt.tight_layout()
plt.show()

# ── (b) TreeSHAP ───────────────────────────────────────────────────────────
rf_explainer = shap.TreeExplainer(rf_best)
shap_rf_vals = rf_explainer.shap_values(X_val_rf)

# Normalise to (N, F) per output — handles both SHAP return formats:
#   • Legacy / list format:  list of 2 arrays each (N, F)
#   • New 3-D format:        single array (N, F, n_outputs)
shap_rf_vals = np.array(shap_rf_vals)  # unify to ndarray

if shap_rf_vals.ndim == 3:
    # Shape is (N, F, 2) — slice along last axis
    shap_rf_gu   = shap_rf_vals[:, :, 0]   # (N, F)
    shap_rf_camp = shap_rf_vals[:, :, 1]   # (N, F)
elif shap_rf_vals.ndim == 2:
    # Shape is (2, N, F) when converted from a 2-element list
    shap_rf_gu   = shap_rf_vals[0]         # (N, F)
    shap_rf_camp = shap_rf_vals[1]         # (N, F)
else:
    raise ValueError(f"Unexpected shap_values shape: {shap_rf_vals.shape}")

# Sanity check — both arrays must be (N, F) matching X_val_rf
assert shap_rf_gu.shape == X_val_rf.shape, (
    f"Shape mismatch: shap_rf_gu {shap_rf_gu.shape} vs X_val_rf {X_val_rf.shape}"
)

print('\n[RF] SHAP Beeswarm — GU:')
shap.summary_plot(shap_rf_gu, X_val_rf, feature_names=names_filtered, max_display=20)

print('\n[RF] SHAP Beeswarm — cAMP:')
shap.summary_plot(shap_rf_camp, X_val_rf, feature_names=names_filtered, max_display=20)

print('\n[RF] Bias Score — High GU & Low cAMP:')
rf_bias_df = bias_score_plot(
    shap_rf_gu, shap_rf_camp,
    names_filtered, top_n=20,
    title='Random Forest — Bias Score (High GU & Low cAMP)'
)
display(rf_bias_df.head(10))

### 6.3 TreeSHAP + Feature Importance — XGBoost

In [ ]:
best_fold_xgb = np.argmin([
    (m['gu']['RMSE'] + m['camp']['RMSE']) / 2 for m in xgb_fold_metrics
])
xgb_best, tr_idx_xgb, val_idx_xgb = xgb_fold_models[best_fold_xgb]

X_tr_xgb  = X_filtered[tr_idx_xgb]
X_val_xgb = X_filtered[val_idx_xgb]

# ── (a) Built-in feature importance ───────────────────────────────────────
# multi_output_tree does NOT support gain/total_gain importance.
# Use 'weight' (split count) which IS supported for multi-target trees.
booster = xgb_best.get_booster()
scores = booster.get_score(importance_type='weight')  # dict {feature_name: score}

# Map scores back to the filtered feature index order.
n_feats = X_tr_xgb.shape[1]
fi_xgb = np.zeros(n_feats)
for fname, score in scores.items():
    idx = int(fname[1:])   # strip leading 'f'
    if idx < n_feats:
        fi_xgb[idx] = score

# Normalise to [0, 1]
fi_xgb = fi_xgb / fi_xgb.sum() if fi_xgb.sum() > 0 else fi_xgb

top20_xgb = np.argsort(fi_xgb)[::-1][:20]
plt.figure(figsize=(10, 6))
plt.barh([names_filtered[i] for i in top20_xgb][::-1],
          fi_xgb[top20_xgb][::-1], color='darkorange')
plt.xlabel('Weight Importance (normalised split count)')
plt.title('XGBoost Feature Importance (Multi-Output, weight)')
plt.tight_layout()
plt.show()

# ── (b) KernelSHAP — per output ───────────────────────────────────────────
# shap.TreeExplainer does NOT support multi_output_tree (vector-leaf trees).
# KernelExplainer is model-agnostic and works correctly here.
# We subsample the validation set to keep runtime tractable.
MAX_KERNEL_SAMPLES = min(100, X_val_xgb.shape[0])
X_val_xgb_sub = X_val_xgb[:MAX_KERNEL_SAMPLES]

print("Computing KernelSHAP for XGBoost (this may take a few minutes)...")

def predict_gu(x):   return xgb_best.predict(x)[:, 0]
def predict_camp(x): return xgb_best.predict(x)[:, 1]

# k-means background summary keeps KernelSHAP tractable
bg = shap.kmeans(X_tr_xgb, 30)

print("  → Output 1/2: GU")
ex_gu   = shap.KernelExplainer(predict_gu,   bg)
shap_xgb_gu   = ex_gu.shap_values(X_val_xgb_sub, nsamples=200)

print("  → Output 2/2: cAMP")
ex_camp = shap.KernelExplainer(predict_camp, bg)
shap_xgb_camp = ex_camp.shap_values(X_val_xgb_sub, nsamples=200)

shap_xgb_gu   = np.array(shap_xgb_gu)
shap_xgb_camp = np.array(shap_xgb_camp)

# Sanity check
assert shap_xgb_gu.shape == X_val_xgb_sub.shape, (
    f"Shape mismatch: shap_xgb_gu {shap_xgb_gu.shape} vs X_val_xgb_sub {X_val_xgb_sub.shape}"
)

print('\n[XGB] SHAP Beeswarm — GU:')
shap.summary_plot(shap_xgb_gu, X_val_xgb_sub, feature_names=names_filtered, max_display=20)

print('\n[XGB] SHAP Beeswarm — cAMP:')
shap.summary_plot(shap_xgb_camp, X_val_xgb_sub, feature_names=names_filtered, max_display=20)

print('\n[XGB] Bias Score — High GU & Low cAMP:')
xgb_bias_df = bias_score_plot(
    shap_xgb_gu, shap_xgb_camp,
    names_filtered, top_n=20,
    title='XGBoost — Bias Score (High GU & Low cAMP)'
)
display(xgb_bias_df.head(10))

## 7. Consensus Bias Score Across Models

Aggregate MACCS-level bias scores from SVR, RF, and XGBoost to find features that **all models agree** drive high GU + low cAMP.

In [ ]:
def get_bias_vector(shap_gu, shap_camp):
    """Mean SHAP GU - mean SHAP cAMP per feature."""
    return shap_gu.mean(axis=0) - shap_camp.mean(axis=0)

bias_svr  = get_bias_vector(np.array(shap_svr_gu),   np.array(shap_svr_camp))
bias_rf   = get_bias_vector(shap_rf_gu,   shap_rf_camp)
bias_xgb  = get_bias_vector(shap_xgb_gu,  shap_xgb_camp)

# Normalise each to [-1, 1] before averaging
def normalise(v):
    m = np.abs(v).max()
    return v / m if m > 0 else v

consensus_bias = (normalise(bias_svr) + normalise(bias_rf) + normalise(bias_xgb)) / 3.0

top_n = 25
idx_top = np.argsort(np.abs(consensus_bias))[::-1][:top_n]
names_top = [ap_names_filtered[i] for i in idx_top]
vals_top  = consensus_bias[idx_top]

colors = ['#1976D2' if v > 0 else '#D32F2F' for v in vals_top]
plt.figure(figsize=(11, 7))
plt.barh(names_top[::-1], vals_top[::-1], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Normalised Consensus Bias Score')
plt.title('Consensus Bias Score (SVR + RF + XGBoost)\n↑ = High Glucose Uptake & Low cAMP', fontsize=13)
plt.tight_layout()
plt.show()

consensus_df = pd.DataFrame({
    'Feature':        names_top,
    'ConsensusScore': vals_top,
    'SVR_bias':  bias_svr[idx_top],
    'RF_bias':   bias_rf[idx_top],
    'XGB_bias':  bias_xgb[idx_top]
}).sort_values('ConsensusScore', ascending=False)

print('Top features driving HIGH GU & LOW cAMP (positive score):')
display(consensus_df[consensus_df['ConsensusScore'] > 0].head(10))
print('\nTop features that hinder the desired outcome (negative score):')
display(consensus_df[consensus_df['ConsensusScore'] < 0].head(10))

## 8. Atom Pair Bit Interpretation Helper

Decode top-scoring Atom Pair bit indices back to their atom-type/distance encoding using RDKit.

In [ ]:
# ── Section 12: Atom Pair Interpretation (unhashed FP version) ────────────
# With unhashed fingerprints the feature name IS the pair code,
# so decoding is a direct lookup — no co-occurrence scanning required.

def decode_pair_code(pair_code):
    """
    Corrected RDKit atom pair bit layout (per rdkit/Code/DataStructs/AtomPairs.h):
      bits  0–9  : atom1 descriptor
                    bits 0–2  : atomic num bucket  (3 bits)
                    bits 3–5  : num heavy neighbours (3 bits)  
                    bits 6–7  : num pi electrons    (2 bits)
                    bits 8–9  : chirality           (2 bits)
      bits 10–12 : shortest-path distance          (3 bits, values 1–7)
      bits 13–22 : atom2 descriptor                (same 10-bit layout)
    """
    atom1_code =  pair_code        & 0x3FF   # bits  0–9  (10 bits)
    distance   = (pair_code >> 10) & 0x07    # bits 10–12 ( 3 bits) ← was 0x1F (5 bits)
    atom2_code = (pair_code >> 13) & 0x3FF   # bits 13–22 (10 bits) ← was >> 15

    def atom_code_to_str(code):
        atomic_num =  code       & 0x7
        n_branches = (code >> 3) & 0x7
        n_pi       = (code >> 6) & 0x3
        atomic_map = {0:'C', 1:'N', 2:'O', 3:'S', 4:'F', 5:'Cl/Br', 6:'P', 7:'other'}
        element    = atomic_map.get(atomic_num, '?')
        return f"{element}(nbrs={n_branches}, pi={n_pi})"

    a1 = atom_code_to_str(atom1_code)
    a2 = atom_code_to_str(atom2_code)
    return f"{a1}  —[{distance} bonds]—  {a2}"


def describe_ap_feature(feature_name):
    """
    Decode a feature name of the form 'AP_<pair_code>' directly.
    With unhashed fingerprints the suffix IS the RDKit pair code integer.
    """
    pair_code = int(feature_name.split('_', 1)[1])
    return decode_pair_code(pair_code)


# Apply directly — no molecule scanning needed
consensus_df['ChemicalMeaning'] = consensus_df['Feature'].apply(describe_ap_feature)

# ── Display ────────────────────────────────────────────────────────────────
print('Top positive bias features (High GU & Low cAMP):')
display(
    consensus_df[consensus_df['ConsensusScore'] > 0]
    [['Feature', 'ChemicalMeaning', 'ConsensusScore']]
    .head(10)
)

print('\nTop negative bias features:')
display(
    consensus_df[consensus_df['ConsensusScore'] < 0]
    [['Feature', 'ChemicalMeaning', 'ConsensusScore']]
    .head(10)
)

# ── Sanity check: verify a few decodings against RDKit directly ───────────
print('\n── Verification: re-encode top 5 features back to pair codes ────')
for feat in consensus_df['Feature'].head(5):
    pair_code = int(feat.split('_', 1)[1])
    meaning   = decode_pair_code(pair_code)
    print(f"  {feat:>15}  →  {meaning}")

In [ ]:
# ── Confirm atom descriptor field order from raw a1/a2 values ─────────────
# Take AP_1590600: a1_raw=328, a2_raw=194
# Decode both field order hypotheses and check which is chemically sensible

def try_both_field_orders(raw_code, label=""):
    bin_str = format(raw_code, '010b')
    
    # Hypothesis A — previous versions: atomic_num=bits0-2, degree=bits3-5, pi=bits6-7
    h_a_elem   =  raw_code       & 0x7
    h_a_degree = (raw_code >> 3) & 0x7
    h_a_pi     = (raw_code >> 6) & 0x3

    # Hypothesis B — RDKit AtomPairs.cpp order: pi=bits0-2, degree=bits3-5, atomic_num=bits6-8
    h_b_pi     =  raw_code       & 0x7
    h_b_degree = (raw_code >> 3) & 0x7
    h_b_elem   = (raw_code >> 6) & 0x7

    amap = {0:'C', 1:'N', 2:'O', 3:'S', 4:'F', 5:'Cl/Br', 6:'P', 7:'other'}

    print(f"\n  {label}  raw={raw_code}  bin={bin_str}")
    print(f"    Hypothesis A (elem|degree|pi): {amap.get(h_a_elem,'?')}  degree={h_a_degree}  pi={h_a_pi}")
    print(f"    Hypothesis B (pi|degree|elem): {amap.get(h_b_elem,'?')}  degree={h_b_degree}  pi={h_b_pi}")

# AP_1590600: a1=328, a2=194
try_both_field_orders(328, "AP_1590600 atom1")
try_both_field_orders(194, "AP_1590600 atom2")

# AP_574535: a1=71, a2=70  (small values, easier to reason about)
try_both_field_orders(71,  "AP_574535  atom1")
try_both_field_orders(70,  "AP_574535  atom2")

# AP_689219: a1=67, a2=84
try_both_field_orders(67,  "AP_689219  atom1")
try_both_field_orders(84,  "AP_689219  atom2")

# AP_558115: a1=35, a2=68
try_both_field_orders(35,  "AP_558115  atom1")
try_both_field_orders(68,  "AP_558115  atom2")

In [ ]:
def decode_pair_code_final(pair_code):
    """
    Final decoder — field order confirmed from raw bit inspection.
    RDKit AtomPairs.cpp atom invariant layout:
      bits 0-2 : num pi electrons  (0-3, capped)
      bits 3-5 : degree / heavy neighbours (0-7, capped)
      bits 6-8 : atomic number bucket (0-7)
    Distance:
      bits 10-12 : shortest-path length (1-7)
    """
    atom1_code =  pair_code        & 0x3FF   # bits 0-9
    distance   = (pair_code >> 10) & 0x07    # bits 10-12
    atom2_code = (pair_code >> 13) & 0x3FF   # bits 13-22

    def atom_code_to_str(code):
        n_pi    =  code       & 0x7   # bits 0-2
        degree  = (code >> 3) & 0x7   # bits 3-5
        atype   = (code >> 6) & 0x7   # bits 6-8
        amap    = {0:'C', 1:'N', 2:'O', 3:'S', 4:'F', 5:'Cl/Br', 6:'P', 7:'other'}
        return f"{amap.get(atype,'?')}(degree={degree}, pi={n_pi})"

    a1 = atom_code_to_str(atom1_code)
    a2 = atom_code_to_str(atom2_code)
    return f"{a1}  —[{distance} bonds]—  {a2}"

# Apply to consensus_df
consensus_df['ChemicalMeaning'] = consensus_df['Feature'].apply(
    lambda name: decode_pair_code_final(int(name.split('_', 1)[1]))
)

print('Top positive bias features (High GU & Low cAMP):')
display(
    consensus_df[consensus_df['ConsensusScore'] > 0]
    [['Feature', 'ChemicalMeaning', 'ConsensusScore']]
    .head(10)
)

print('\nTop negative bias features:')
display(
    consensus_df[consensus_df['ConsensusScore'] < 0]
    [['Feature', 'ChemicalMeaning', 'ConsensusScore']]
    .head(10)
)

In [ ]:
# ══ Atom Pair Fragment Visualisation ══════════════════════════════════════
from rdkit.Chem import rdMolDescriptors as rdMD
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem import rdmolops
from PIL import Image
import io, math

TOP_N_BITS = 10

# ── 1. Build name → original bit index lookup dict ────────────────────────
# ap_names_filtered[i] = 'AP_<ap_original_indices[i]>'
# The integer suffix in the feature name IS the original 2048-bit index.
ap_name_to_orig = {name: orig_idx
                   for name, orig_idx in zip(ap_names_filtered, ap_original_indices)}

# ── 2. Positive-bias AP bits from consensus_df ────────────────────────────
pos_bits_df = consensus_df[consensus_df['ConsensusScore'] > 0].head(TOP_N_BITS).copy()

def resolve_orig_bit(feature_name):
    """
    Resolve a feature name to its original 2048-bit index.
    Strategy 1: look up in ap_name_to_orig (exact match).
    Strategy 2: the feature name is 'AP_<N>' — the suffix N is already
                the original bit index, because ap_names was built as
                f'AP_{i}' where i is the position in the full 2048-bit array.
    Strategy 3: feature name is a plain integer string (fallback).
    Returns int or NaN.
    """
    # Strategy 1: exact dict lookup
    if feature_name in ap_name_to_orig:
        return ap_name_to_orig[feature_name]
    # Strategy 2: parse 'AP_<N>' suffix directly
    if isinstance(feature_name, str) and feature_name.startswith('AP_'):
        try:
            return int(feature_name.split('_', 1)[1])
        except ValueError:
            pass
    # Strategy 3: plain integer string
    try:
        return int(feature_name)
    except (ValueError, TypeError):
        pass
    return float('nan')

pos_bits_df['OrigBitIdx'] = pos_bits_df['Feature'].apply(resolve_orig_bit)

# Report any that still couldn't be resolved
missing = pos_bits_df[pos_bits_df['OrigBitIdx'].isna()]
if not missing.empty:
    print(f'[!] {len(missing)} feature(s) could not be resolved — they will be skipped:')
    print(f'    {missing["Feature"].tolist()}')
    pos_bits_df = pos_bits_df[pos_bits_df['OrigBitIdx'].notna()].copy()

pos_bits_df['OrigBitIdx'] = pos_bits_df['OrigBitIdx'].astype(int)

# Clamp to valid range (guard against any off-by-one)
pos_bits_df = pos_bits_df[pos_bits_df['OrigBitIdx'].between(0, AP_NBITS - 1)].copy()

print('Top positive-bias Atom Pair bits (original bit index in 2048-bit space):')
for _, row in pos_bits_df.iterrows():
    print(f'  {row["Feature"]}  (orig bit {row["OrigBitIdx"]:>4d})  '
          f'Consensus bias = {row["ConsensusScore"]:.4f}')


# ── 3. Atom code (replicates RDKit internal, no public API needed) ─────────
def _atom_code(atom):
    """
    Replicate RDKit's internal atom-pair atom code:
        bits 0-4  : atomic number (capped at 31)
        bits 5-6  : number of pi electrons (capped at 3)
        bits 7-8  : degree (capped at 3)
    """
    atomic_num = min(atom.GetAtomicNum(), 31)

    n_pi = 0
    for bond in atom.GetBonds():
        bt = bond.GetBondTypeAsDouble()
        if bt == 2.0:
            n_pi += 2
        elif bt == 3.0:
            n_pi += 4
        elif bond.GetIsAromatic():
            n_pi += 1
    n_pi = min(n_pi, 3)

    degree = min(atom.GetDegree(), 3)
    return atomic_num | (n_pi << 5) | (degree << 7)


# ── 4. Find the atom pair responsible for a hashed AP bit ─────────────────
def find_atom_pair_for_bit(mol, orig_bit_idx, n_bits=AP_NBITS):
    """
    Scan all atom pairs. Replicate RDKit's hash:
        (ac_lo | (dist << 10) | (ac_hi << 14)) % n_bits
    Returns (atom_i, atom_j, distance, label) or None.
    """
    n_atoms = mol.GetNumAtoms()
    for ai in range(n_atoms):
        for aj in range(ai + 1, n_atoms):
            try:
                path = rdmolops.GetShortestPath(mol, ai, aj)
            except Exception:
                continue
            dist = len(path) - 1
            if dist < 1:
                continue

            ac_i = _atom_code(mol.GetAtomWithIdx(ai))
            ac_j = _atom_code(mol.GetAtomWithIdx(aj))
            ac_lo, ac_hi = sorted([ac_i, ac_j])

            pair_hash = (ac_lo | (dist << 10) | (ac_hi << 14)) % n_bits

            if pair_hash == orig_bit_idx:
                sym_i = mol.GetAtomWithIdx(ai).GetSymbol()
                sym_j = mol.GetAtomWithIdx(aj).GetSymbol()
                label = (f'{sym_i} (atom {ai}) ···  {sym_j} (atom {aj})  '
                         f'dist={dist}  [AP bit {orig_bit_idx}]')
                return ai, aj, dist, label
    return None


# ── 5. Draw molecule with highlighted atom pair ────────────────────────────
def draw_atom_pair_highlight(mol, atom_i, atom_j, mol_size=(300, 230)):
    """Return PIL Image: atom_i blue, atom_j orange, path green."""
    try:
        path = list(rdmolops.GetShortestPath(mol, atom_i, atom_j))
    except Exception:
        path = [atom_i, atom_j]

    highlight_atoms = path
    highlight_bonds = []
    for k in range(len(path) - 1):
        b = mol.GetBondBetweenAtoms(path[k], path[k+1])
        if b:
            highlight_bonds.append(b.GetIdx())

    atom_colors = {idx: (0.75, 0.92, 0.75) for idx in path}
    atom_colors[atom_i] = (0.20, 0.50, 0.90)
    atom_colors[atom_j] = (1.00, 0.60, 0.10)
    bond_colors = {b: (0.55, 0.80, 0.55) for b in highlight_bonds}

    try:
        drawer = rdMolDraw2D.MolDraw2DSVG(mol_size[0], mol_size[1])
        drawer.drawOptions().addAtomIndices = False
        drawer.DrawMolecule(
            mol,
            highlightAtoms=highlight_atoms,
            highlightAtomColors=atom_colors,
            highlightBonds=highlight_bonds,
            highlightBondColors=bond_colors
        )
        drawer.FinishDrawing()
        svg = drawer.GetDrawingText()
        try:
            import cairosvg
            return Image.open(io.BytesIO(cairosvg.svg2png(bytestring=svg.encode())))
        except ImportError:
            return Draw.MolToImage(mol, size=mol_size,
                                   highlightAtoms=highlight_atoms,
                                   highlightColor=(0.2, 0.6, 1.0))
    except Exception as e:
        print(f'  [draw error] {e}')
        return Draw.MolToImage(mol, size=mol_size)


# ── 6. Build images and descriptions ──────────────────────────────────────
imgs, legends, descriptions = [], [], []

for _, row in pos_bits_df.iterrows():
    orig_bit = int(row['OrigBitIdx'])
    found = False

    for smi in df['SMILES']:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        if smiles_to_atom_pair(smi)[orig_bit] == 0:
            continue

        result = find_atom_pair_for_bit(mol, orig_bit)
        if result is not None:
            ai, aj, dist, label = result
            img = draw_atom_pair_highlight(mol, ai, aj)
        else:
            img   = Draw.MolToImage(mol, size=(300, 230))
            label = f'AP bit {orig_bit} — pair not decoded'

        imgs.append(img)
        legends.append(f'Bit {orig_bit}\n(bias={row["ConsensusScore"]:.3f})')
        descriptions.append(label)
        found = True
        break

    if not found:
        print(f'  [!] No molecule found in dataset for AP bit {orig_bit}')


# ── 7. Render grid ─────────────────────────────────────────────────────────
if imgs:
    ncols = min(5, len(imgs))
    nrows = math.ceil(len(imgs) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.8, nrows * 3.8))
    axes = np.array(axes).flatten()

    for ax, img, leg, desc in zip(axes, imgs, legends, descriptions):
        ax.imshow(img)
        ax.set_title(leg, fontsize=9, fontweight='bold')
        ax.set_xlabel(desc, fontsize=7)
        ax.axis('off')
    for ax in axes[len(imgs):]:
        ax.axis('off')

    plt.suptitle(
        'Top Positive-Bias Atom Pair Fragments\n'
        'Blue = atom i  |  Orange = atom j  |  Green = topological path',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()
else:
    print('No images generated — verify AP_NBITS matches the fingerprint used.')


# ── 8. Summary table ────────────────────────────────────────────────────────
pos_bits_df = pos_bits_df.copy()
pos_bits_df['AtomPairDesc'] = (descriptions + [''] * TOP_N_BITS)[:len(pos_bits_df)]
print('\nTop positive-bias Atom Pair feature table:')
display(
    pos_bits_df[['Feature', 'OrigBitIdx', 'ConsensusScore',
                 'SVR_bias', 'RF_bias', 'XGB_bias',
                 'AtomPairDesc']].reset_index(drop=True)
)